# Project 96 — Local Documentation Writer
## Source Code → API Docs → Tutorials → Reference

**Stack:** LangChain · Ollama · Pydantic · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic

## Step 1 — Source Code to Document

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import json

llm = ChatOllama(model="qwen3:8b", temperature=0.3)

source_files = [
    {
        "name": "auth.py",
        "code": '''
import jwt
from datetime import datetime, timedelta
from hashlib import sha256

class AuthService:
    def __init__(self, secret_key: str, token_expiry: int = 3600):
        self.secret_key = secret_key
        self.token_expiry = token_expiry

    def hash_password(self, password: str) -> str:
        return sha256(password.encode()).hexdigest()

    def create_token(self, user_id: int, roles: list[str]) -> str:
        payload = {
            "sub": user_id,
            "roles": roles,
            "exp": datetime.utcnow() + timedelta(seconds=self.token_expiry),
            "iat": datetime.utcnow(),
        }
        return jwt.encode(payload, self.secret_key, algorithm="HS256")

    def verify_token(self, token: str) -> dict | None:
        try:
            return jwt.decode(token, self.secret_key, algorithms=["HS256"])
        except jwt.ExpiredSignatureError:
            return None
        except jwt.InvalidTokenError:
            return None

    def refresh_token(self, token: str) -> str | None:
        payload = self.verify_token(token)
        if payload:
            return self.create_token(payload["sub"], payload["roles"])
        return None
'''
    },
    {
        "name": "cache.py",
        "code": '''
from typing import Any, Optional
from time import time

class LRUCache:
    def __init__(self, max_size: int = 100, ttl: int = 300):
        self.max_size = max_size
        self.ttl = ttl
        self.cache: dict[str, tuple[Any, float]] = {}
        self.access_order: list[str] = []

    def get(self, key: str) -> Optional[Any]:
        if key in self.cache:
            value, timestamp = self.cache[key]
            if time() - timestamp < self.ttl:
                self.access_order.remove(key)
                self.access_order.append(key)
                return value
            else:
                del self.cache[key]
                self.access_order.remove(key)
        return None

    def set(self, key: str, value: Any) -> None:
        if key in self.cache:
            self.access_order.remove(key)
        elif len(self.cache) >= self.max_size:
            oldest = self.access_order.pop(0)
            del self.cache[oldest]
        self.cache[key] = (value, time())
        self.access_order.append(key)

    def invalidate(self, key: str) -> bool:
        if key in self.cache:
            del self.cache[key]
            self.access_order.remove(key)
            return True
        return False

    def stats(self) -> dict:
        return {"size": len(self.cache), "max_size": self.max_size, "ttl": self.ttl}
'''
    },
]
print(f"Source files to document: {len(source_files)}")

## Step 2 — Generate API Reference

In [ ]:
class MethodDoc(BaseModel):
    name: str
    signature: str
    description: str
    parameters: list[str]
    returns: str
    raises: list[str] = Field(default_factory=list)
    example: str

class ClassDoc(BaseModel):
    class_name: str
    module: str
    description: str
    constructor_params: list[str]
    methods: list[MethodDoc]
    usage_example: str

documenter = llm.with_structured_output(ClassDoc)

api_docs = []
for src in source_files:
    doc = documenter.invoke(
        f"Generate comprehensive API documentation for this class:\n\n"
        f"Module: {src['name']}\n```python\n{src['code']}\n```"
    )
    api_docs.append(doc)
    print(f"\n{'='*50}")
    print(f"## {doc.class_name} ({doc.module})")
    print(f"{doc.description}")
    print(f"\nMethods: {len(doc.methods)}")
    for m in doc.methods:
        print(f"  • {m.name}({', '.join(m.parameters[:3])}) → {m.returns}")

## Step 3 — Generate Tutorial

In [ ]:
tutorial_prompt = ChatPromptTemplate.from_messages([
    ("system", "Write a beginner-friendly tutorial for this class. Include:\n"
     "1. Installation steps\n2. Quick start example\n3. Common use cases\n"
     "4. Error handling\n5. Best practices\n\nUse Markdown with code blocks."),
    ("human", "Class: {class_name}\nAPI: {api}\n\nTutorial:")
])
tutorial_chain = tutorial_prompt | llm | StrOutputParser()

for doc in api_docs:
    tutorial = tutorial_chain.invoke({
        "class_name": doc.class_name,
        "api": json.dumps(doc.model_dump(), indent=2),
    })
    print(f"\n{'='*50}")
    print(f"TUTORIAL: {doc.class_name}")
    print(f"{'='*50}")
    print(tutorial[:600])

## Step 4 — Generate Markdown Files

In [ ]:
from pathlib import Path
Path("sample_data/docs").mkdir(parents=True, exist_ok=True)

for doc in api_docs:
    content = f"# {doc.class_name}\n\n{doc.description}\n\n"
    content += f"## Constructor\n\n"
    content += f"Parameters: {', '.join(doc.constructor_params)}\n\n"
    content += f"## Methods\n\n"
    for m in doc.methods:
        content += f"### `{m.signature}`\n\n"
        content += f"{m.description}\n\n"
        content += f"**Parameters:**\n"
        for p in m.parameters:
            content += f"- {p}\n"
        content += f"\n**Returns:** {m.returns}\n\n"
        if m.example:
            content += f"**Example:**\n```python\n{m.example}\n```\n\n"

    content += f"## Quick Start\n\n```python\n{doc.usage_example}\n```\n"

    filepath = f"sample_data/docs/{doc.module.replace('.py','')}.md"
    Path(filepath).write_text(content)
    print(f"✓ {filepath} ({len(content)} chars)")

## What You Learned
- **Automated API documentation** from source code
- **Tutorial generation** with examples
- **Markdown export** for documentation sites
- **Structured doc models** for consistent output